In [16]:
import numpy as np, pandas as pd, joblib, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)

LABELS = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

train = pd.read_parquet(DATA / "train.parquet")
val   = pd.read_parquet(DATA / "val.parquet")
print(train.shape, val.shape)

(127431, 8) (15929, 8)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1,2), max_features=200_000,
                      sublinear_tf=True, strip_accents="unicode")

t = time.time()
Xtr = vec.fit_transform(train.text)   # fit on train ONLY
Xva = vec.transform(val.text)
print(Xtr.shape, f"{time.time()-t:.0f}s")

(127431, 200000) 57s


In [3]:
from sklearn.linear_model import LogisticRegression

models = {}
t = time.time()
for l in LABELS:
    clf = LogisticRegression(C=4, class_weight="balanced", max_iter=1000)
    clf.fit(Xtr, train[l])
    models[l] = clf
    print(l, "done", f"{time.time()-t:.0f}s")

toxic done 4s
severe_toxic done 11s
obscene done 14s
threat done 18s
insult done 23s
identity_hate done 32s


In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_probs = np.column_stack([models[l].predict_proba(Xva)[:,1] for l in LABELS])

rows = []
for i, l in enumerate(LABELS):
    rows.append({"label": l,
                 "roc_auc": roc_auc_score(val[l], val_probs[:,i]),
                 "pr_auc":  average_precision_score(val[l], val_probs[:,i]),
                 "positives": int(val[l].sum())})
res = pd.DataFrame(rows).round(3)
print(res)
print("Mean ROC-AUC:", res.roc_auc.mean().round(4))

           label  roc_auc  pr_auc  positives
0          toxic    0.971   0.863       1517
1   severe_toxic    0.982   0.437        156
2        obscene    0.982   0.873        876
3         threat    0.993   0.556         39
4         insult    0.975   0.755        797
5  identity_hate    0.970   0.375        146
Mean ROC-AUC: 0.9788


In [5]:
def score(texts):
    X = vec.transform(texts)
    return pd.DataFrame({l: models[l].predict_proba(X)[:,1] for l in LABELS},
                        index=texts).round(2)

score(["Have a great day!",
       "You are an idiot",
       "I will find you and hurt you",
       "I am a gay man",
       "I am a Muslim woman"])

,toxic,severe_toxic,obscene,threat,insult,identity_hate
Have a great day!,0.10,0.01,0.01,0.00,0.01,0.02
You are an idiot,1.00,0.11,1.00,0.00,1.00,0.39
I will find you and hurt you,0.95,0.01,0.15,0.86,0.26,0.05
I am a gay man,0.97,0.07,0.18,0.01,0.10,0.83
I am a Muslim woman,0.79,0.03,0.19,0.01,0.03,0.96


In [6]:
joblib.dump({"vec": vec, "models": models, "labels": LABELS},
            MODELS / "baseline_tfidf.joblib")
np.save(MODELS / "baseline_val_probs.npy", val_probs)
print("saved")

saved
